In [0]:
# Week 06 - Data Quality Checks
# GridPulse Campus Energy Command Center

print("Week 06 - Data Quality Checks")

Week 06 - Data Quality Checks


In [0]:
# DQ-01: Required field NULL checks

from pyspark.sql.functions import col

consumption_df = spark.table(
    "workspace.gridpulse_silver.consumption_silver"
)

required_columns = [
    "reading_id",
    "meter_id",
    "reading_ts",
    "energy_kwh"
]

print("=== DQ-01: REQUIRED FIELD NULL CHECK ===")

for column_name in required_columns:
    failed_count = consumption_df.filter(
        col(column_name).isNull()
    ).count()

    print(f"{column_name:20} | Failed records: {failed_count}")

=== DQ-01: REQUIRED FIELD NULL CHECK ===
reading_id           | Failed records: 0
meter_id             | Failed records: 0
reading_ts           | Failed records: 0
energy_kwh           | Failed records: 0


In [0]:
# DQ-02: Duplicate reading_id check

print("=== DQ-02: DUPLICATE READING CHECK ===")

duplicate_readings = (
    consumption_df
    .groupBy("reading_id")
    .count()
    .filter("count > 1")
)

duplicate_count = duplicate_readings.count()

print(f"Duplicate reading IDs: {duplicate_count}")

if duplicate_count > 0:
    display(duplicate_readings)
else:
    print("No duplicate reading IDs found")

=== DQ-02: DUPLICATE READING CHECK ===
Duplicate reading IDs: 0
No duplicate reading IDs found


In [0]:
# DQ-03: Negative energy consumption check

print("=== DQ-03: NEGATIVE ENERGY CHECK ===")

negative_energy = consumption_df.filter(
    col("energy_kwh") < 0
)

negative_count = negative_energy.count()

print(f"Negative energy records: {negative_count}")

if negative_count > 0:
    display(negative_energy.limit(20))
else:
    print("No negative energy records found")

=== DQ-03: NEGATIVE ENERGY CHECK ===
Negative energy records: 0
No negative energy records found


In [0]:
# DQ-04: Future reading timestamp check

from pyspark.sql.functions import current_timestamp

print("=== DQ-04: FUTURE TIMESTAMP CHECK ===")

future_readings = consumption_df.filter(
    col("reading_ts") > current_timestamp()
)

future_count = future_readings.count()

print(f"Future timestamp records: {future_count}")

if future_count > 0:
    display(future_readings.limit(20))
else:
    print("No future timestamp records found")

=== DQ-04: FUTURE TIMESTAMP CHECK ===
Future timestamp records: 0
No future timestamp records found


In [0]:
# DQ-05: Consumption meter reference check

print("=== DQ-05: METER REFERENCE CHECK ===")

meters_df = spark.table(
    "workspace.gridpulse_silver.meters_silver"
)

missing_meter_records = (
    consumption_df
    .join(
        meters_df.select("meter_id").distinct(),
        on="meter_id",
        how="left_anti"
    )
)

missing_meter_count = missing_meter_records.count()

print(f"Consumption records with missing meter master: {missing_meter_count}")

if missing_meter_count > 0:
    display(missing_meter_records.limit(20))
else:
    print("All consumption meter IDs exist in the meter master")

=== DQ-05: METER REFERENCE CHECK ===
Consumption records with missing meter master: 0
All consumption meter IDs exist in the meter master


In [0]:
# Week 06 - DQ Results Summary

dq_results = [
    ("DQ-01", "Required fields NULL check", 0),
    ("DQ-02", "Duplicate reading_id check", 0),
    ("DQ-03", "Negative energy_kwh check", 0),
    ("DQ-04", "Future reading_ts check", 0),
    ("DQ-05", "Missing meter reference check", 0)
]

dq_summary_df = spark.createDataFrame(
    dq_results,
    ["Rule_ID", "DQ_Rule", "Failed_Count"]
)

display(dq_summary_df)

Rule_ID,DQ_Rule,Failed_Count
DQ-01,Required fields NULL check,0
DQ-02,Duplicate reading_id check,0
DQ-03,Negative energy_kwh check,0
DQ-04,Future reading_ts check,0
DQ-05,Missing meter reference check,0


In [0]:
# Week 06 - Failed Records Evidence

print("=== WEEK 06 FAILED RECORDS SAMPLE ===")
print("DQ-01 Required fields NULL        : 0 failed records")
print("DQ-02 Duplicate reading_id        : 0 failed records")
print("DQ-03 Negative energy_kwh         : 0 failed records")
print("DQ-04 Future reading_ts           : 0 failed records")
print("DQ-05 Missing meter reference     : 0 failed records")
print()
print("No failed records were found in the current Silver dataset.")

=== WEEK 06 FAILED RECORDS SAMPLE ===
DQ-01 Required fields NULL        : 0 failed records
DQ-02 Duplicate reading_id        : 0 failed records
DQ-03 Negative energy_kwh         : 0 failed records
DQ-04 Future reading_ts           : 0 failed records
DQ-05 Missing meter reference     : 0 failed records

No failed records were found in the current Silver dataset.
